TODO: natively vectorize as many functions as possible to avoid iterations and speed up computation.

In [1]:
from engine import WordleEngine
import numpy as np
from constants import Square
from wordfreq import zipf_frequency


WORD_LEN = 5
LOCALE = 'en'
FREQ_VEC = np.vectorize(lambda word: zipf_frequency(word, LOCALE), otypes=[float])

engine = WordleEngine('words.txt', WORD_LEN)
df = engine._df
patterns = engine._patterns

In [2]:
def convert_str_to_squares(abbrev: str) -> np.ndarray:
    """Convenience wrapper to save myself some copy pasting."""
    squares = []
    for letter in abbrev:
        assert letter in ('g', 'y', 'b')
        if letter == 'g':
            squares.append(Square.GREEN.value)
        elif letter == 'y':
            squares.append(Square.YELLOW.value)
        else:
            squares.append(Square.BLACK.value)
    return np.array(squares, dtype=np.uint8)

In [3]:
import pandas as pd
from cache import is_match

# TODO: implement is_match using pattern look up.

def filter_results(current: pd.DataFrame, guess: str, response: str) -> np.ndarray:
    squares = convert_str_to_squares(response)
    gs_np = df.loc[guess].to_numpy()
    mask = [is_match(gs_np, row.to_numpy(), squares) \
            for _, row in current.iterrows()]
    return np.array(mask)

In [4]:
from scipy.stats import entropy

def get_entropy(current: np.ndarray):
    entrops = np.empty((current.shape[0],), dtype=float)
    for idx in range(current.shape[0]):
        _, counts = np.unique(current[idx, :, :], axis=0, return_counts=True)
        entrops[idx] = entropy(counts / current.shape[1], base=2)
    return entrops

Play a game of Worlde below. Record your guesses and responses as you make them and rerun the following cells to refresh the suggestions.

In [5]:
guesses = [
    'tares', # the optimal starting word
    'nitty',
    'halwa',
    'watch',
]
responses = [
    'ygbbb',
    'bbgbb',
    'ygbyb',
    'ggggg',
]

Iteratively update both suggested solutions and most informative guesses lists.

In [6]:
from math import log2
filtered = df.copy()
suggest = df.copy()
assert patterns is not None
patts = np.copy(patterns)

print(f'Starting entropy = {round(log2(len(filtered)), 3)}')
for guess, response in zip(guesses, responses):
    mask = filter_results(filtered, guess, response)
    filtered = filtered[mask]
    entropy_lost = -log2(np.count_nonzero(mask) / len(mask))
    print(f'\tLost entropy = {round(entropy_lost, 3)}')
    print(f'Remaining entropy = {round(log2(len(filtered)), 3)}')
    patts = patts[:, mask, :]

recc = filtered.copy()
recc['freq'] = FREQ_VEC(recc.index)
recc.sort_values(by='freq', ascending=False, inplace=True)
print('Possible solutions sorted by log freq')
print(recc[:10], '=' * 35, sep='\n')

entrops = get_entropy(patts)
suggest['entropy'] = entrops
print('Most likely informative next guesses sorted by entropy')
suggest.sort_values('entropy', ascending=False, inplace=True)
print(suggest[:10], '=' * 35, sep='\n')

Starting entropy = 13.859
	Lost entropy = 6.825
Remaining entropy = 7.033
	Lost entropy = 2.574
Remaining entropy = 4.459
	Lost entropy = 4.459
Remaining entropy = 0.0
	Lost entropy = -0.0
Remaining entropy = 0.0
Possible solutions sorted by log freq
         0   1    2   3    4  freq
watch  119  97  116  99  104  5.34
Most likely informative next guesses sorted by entropy
        0   1    2    3    4  entropy
aahed  97  97  104  101  100      0.0
aalii  97  97  108  105  105      0.0
aapas  97  97  112   97  115      0.0
aargh  97  97  114  103  104      0.0
aarti  97  97  114  116  105      0.0
abaca  97  98   97   99   97      0.0
abaci  97  98   97   99  105      0.0
aback  97  98   97   99  107      0.0
abacs  97  98   97   99  115      0.0
abaft  97  98   97  102  116      0.0


Once you have the solution, visually ascertain that the produced squares match the game and validate our own function.

In [7]:
expected = 'watch'

for guess, response in zip(guesses, responses):
    squares = engine.lookup_pattern(guess, expected)
    print(guess, ''.join(squares))

tares 🟨🟩⬛⬛⬛
nitty ⬛⬛🟩⬛⬛
halwa 🟨🟩⬛🟨⬛
watch 🟩🟩🟩🟩🟩


A bot playing wordle should always seek the highest entropy word unless it's down to 2 or 3 (less than 2 bits entropy). Then pick the more frequent option.